# ACOS IndoBERT V5.1 — Full Experiment (ROCM + CUDA + Drive Sync)

> Upgrade dari V5: deteksi GPU ganda AMD ROCm / NVIDIA CUDA / CPU, sinkronisasi hasil ke Google Drive agar bisa diunduh, dishare, dan dilanjutkan antar-server, serta EDA + grafik + tabel + penyimpanan komprehensif ala V4.

| Item | V5 | V5.1 |
|---|---|---|
| GPU | MI300X only | ROCm / CUDA / CPU otomatis |
| Output | lokal `results/` | lokal + Drive mirror + manifest |
| EDA/Viz | tidak ada | `acos_id.eda` + 4 PNG + tabel |
| Per-run | hardware_log | + ResultSaver (csv/plots/md/checkpoints) |

```
Sel 1  : Diagnostik GPU dual-backend (ROCM/NVIDIA)
Sel 2  : Konfigurasi + Drive sync flags
Sel 3  : Import & path + auto-sync ala V4 + sync_to_gdrive()
Sel 4  : HardwareMonitor dual-backend
Sel 5  : Backbone & tokenizer
Sel 5b : EDA + visualisasi (tabel + 4 PNG)
Sel 6  : ExperimentGrid preview + grafik
Sel 7  : Persiapan data (cached)
Sel 8  : train_one_run + ResultSaver
Sel 9  : Run semua eksperimen
Sel 10 : Agregasi + grafik perbandingan + tabel
Sel 11 : Hardware summary + REPORT_INDEX + Drive final sync
```


## Sel 1 — Diagnostik GPU & Hardware

In [44]:
# Sel 1: Diagnostik GPU dual-backend (ROCm / CUDA / CPU)
import subprocess, sys, os, time, platform, json, shutil
from datetime import datetime

SESSION_START = datetime.now()
SESSION_START_TS = SESSION_START.strftime('%d%m%Y_%H%M%S')
print(f'Waktu mulai sesi : {SESSION_START.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Python           : {sys.version}')
print(f'Platform         : {platform.platform()}')
print()

import torch
HAS_CUDA = torch.cuda.is_available()
HAS_ROCM = HAS_CUDA and 'rocm' in torch.__version__.lower()
HAS_NVIDIA_SMI = shutil.which('nvidia-smi') is not None
HAS_ROCM_SMI = shutil.which('rocm-smi') is not None or shutil.which('amd-smi') is not None
DEVICE = 'cuda' if HAS_CUDA else 'cpu'
BACKEND = 'rocm' if HAS_ROCM else ('cuda' if HAS_CUDA else 'cpu')

print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {HAS_CUDA}')
print(f'Backend          : {BACKEND}')
print(f'nvidia-smi       : {HAS_NVIDIA_SMI}')
print(f'rocm/amd-smi     : {HAS_ROCM_SMI}')
print(f'Device           : {DEVICE}')
print()

TOTAL_VRAM_GB, GPU_NAME, IS_LARGE_GPU = 0.0, 'CPU', False
if HAS_CUDA:
    n_gpu = torch.cuda.device_count()
    print(f'GPU count        : {n_gpu}')
    for i in range(n_gpu):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f'  GPU[{i}] : {props.name}')
        print(f'          VRAM = {vram_gb:.2f} GB ({props.total_memory:,} bytes)')
        print(f'          SM   = {props.multi_processor_count} multiprocessors')
    props = torch.cuda.get_device_properties(0)
    TOTAL_VRAM_GB = props.total_memory / 1024**3
    GPU_NAME = props.name
    IS_LARGE_GPU = TOTAL_VRAM_GB >= 40.0
# --- V5.1 adaptive tier: SMALL=T4/CPU (<=20GB), MEDIUM=L4 (<=40GB), LARGE=MI300X/A100 (>=40GB) ---
_g = GPU_NAME.lower()
if TOTAL_VRAM_GB <= 0:
    GPU_TIER = 'SMALL'
elif 't4' in _g or TOTAL_VRAM_GB <= 20:
    GPU_TIER = 'SMALL'
elif 'l4' in _g or TOTAL_VRAM_GB <= 40:
    GPU_TIER = 'MEDIUM'
else:
    GPU_TIER = 'LARGE'
print()

# Coba baca utilisasi via rocm-smi lalu nvidia-smi (best-effort)
def _try_smi():
    for cmd in (['rocm-smi', '--showmeminfo', 'vram', '--json'], ['amd-smi', 'metric', '--json'] if HAS_ROCM_SMI else None, ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu', '--format=csv,noheader,nounits'] if HAS_NVIDIA_SMI else None):
        if not cmd: continue
        try:
            out = subprocess.check_output(cmd, stderr=subprocess.DEVNULL, timeout=10, text=True)
            print(f"SMI ({cmd[0]}):")
            print('  ' + out.strip().split(chr(10))[0][:200])
            return True
        except Exception:
            continue
    print('SMI: tidak tersedia - monitoring via torch.cuda')
    return False
SMI_OK = _try_smi()

print()
print('=' * 65)
print(f'GPU Name         : {GPU_NAME}')
print(f'Total VRAM       : {TOTAL_VRAM_GB:.2f} GB')
print(f'BACKEND          : {BACKEND}')
print(f'GPU_TIER         : {GPU_TIER}')
print(f'IS_LARGE_GPU     : {IS_LARGE_GPU}')
print('=' * 65)

_hw_info = {'session_start': SESSION_START.isoformat(), 'session_ts': SESSION_START_TS, 'gpu_name': GPU_NAME, 'total_vram_gb': TOTAL_VRAM_GB, 'backend': BACKEND, 'gpu_tier': GPU_TIER, 'has_rocm': HAS_ROCM, 'smi_ok': SMI_OK, 'torch_version': torch.__version__, 'python_version': sys.version, 'platform': platform.platform()}
print(f'\nHardware info dicatat: {len(_hw_info)} field')


Waktu mulai sesi : 2026-09-26 13:20:03
Python           : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform         : Linux-6.6.122+-x86_64-with-glibc2.39

PyTorch version  : 2.11.0+cu128
CUDA available   : True
Backend          : cuda
nvidia-smi       : True
rocm/amd-smi     : False
Device           : cuda

GPU count        : 1
  GPU[0] : Tesla T4
          VRAM = 14.56 GB (15,637,086,208 bytes)
          SM   = 40 multiprocessors

SMI (nvidia-smi):
  3, 15360, 0

GPU Name         : Tesla T4
Total VRAM       : 14.56 GB
BACKEND          : cuda
GPU_TIER         : SMALL
IS_LARGE_GPU     : False

Hardware info dicatat: 11 field


## Sel 2 — Konfigurasi Dinamis (satu-satunya sel yang perlu diedit)

> Cara pakai: edit dict `CONFIG`, lalu Run sel ini. Semua sel di bawah membaca variabel turunan
> (`DOMAIN`, `EXPERIMENT_*`, `STEP1/2_BATCH_SIZE`, `RUN_MODE`, `DRY_RUN`, `RUN_EPOCHS`, `USE_MODEL_CACHE`, …)
> sehingga tidak perlu edit sel lain. Hasil deteksi Sel 1 (`GPU_TIER`, `BACKEND`, `TOTAL_VRAM_GB`) dipakai
> bila `CONFIG` bernilai `AUTO`/`None`.
>
> | Kunci `CONFIG` | Arti | Contoh |
> |---|---|---|
> | `GPU_TIER_OVERRIDE` | Paksa tier batch/cache: `AUTO` (ikut Sel 1), `SMALL` (T4/CPU), `MEDIUM` (L4), `LARGE` (MI300X/A100) | `'AUTO'` |
> | `DOMAIN`, `BACKBONE` | Domain data (`appsid`) & backbone HF (`indobert`) | — |
> | `EXPERIMENT_EPOCHS` | Daftar epoch grid, mis. 54 run = 3 epoch × (3 rasio + 5 + 10 fold) | `[50, 75, 100]` |
> | `EXPERIMENT_RATIOS` | Split train/dev (test = sisa), level `review_id` anti-bocor | `[(0.8,0.1)]` = 80:10:10 |
> | `EXPERIMENT_CV` | Daftar fold CV, `[]` = nonaktif | `[{'n_splits':5}]` |
> | `RUN_MODE` | Subset eksekusi Sel 9: `all`/`ratio`/`cv` | `'all'` |
> | `DRY_RUN` | `True` = preview tanpa latih; `False` = latih sungguhan | `True` |
> | `RUN_EPOCHS` | Epoch yang dieksekusi; `None` = ikut `EXPERIMENT_EPOCHS` | `None` / `[50]` |
> | `STEP1_BATCH`, `STEP2_BATCH` | Override batch; `None` = adaptif tier | `None` / `16` |
> | `STEP1_LR`, `STEP2_LR`, `MAX_SEQ_LENGTH`, `SEED`, `DO_LOWER_CASE` | LR, panjang sekuen, seed, lower-case IndoBERT | — |
> | `DRIVE_SYNC` | Mirror hasil ke Drive bila termount | `True` |
> | `FORCE_REBUILD_DATA` | `True` = bangun ulang tokenized; `None` = adaptif tier | `None` |
> | `RESUME` | `True` = pakai sesi/checkpoint lama; `None` = adaptif tier | `None` |
>
> Override tanpa edit file via environment (prioritas atas `CONFIG`):
> `ACOS_TIER` (AUTO/SMALL/MEDIUM/LARGE), `ACOS_MODE` (all/ratio/cv), `ACOS_DRY` (1/true/ya),
> `ACOS_EPOCHS` (mis. `"[50]"`), `ACOS_BATCH1`, `ACOS_BATCH2` (mis. `"16"`).
> Contoh cepat: smoke-test T4 → `DRY_RUN=True, RUN_MODE='ratio', RUN_EPOCHS=[50]`;
> full MI300X → `GPU_TIER_OVERRIDE='LARGE', DRY_RUN=False, RUN_MODE='all'`.
>
> Preset tier (bila batch tidak di-override): `LARGE` 96/64 accum 1 worker 4 from-scratch;
> `MEDIUM` 32/24 accum 2 worker 2 cache; `SMALL` 16/8 accum 4 worker 2 cache.
> Batch efektif ≈ `batch × GRAD_ACCUM_STEPS`. `PATIENCE=0` = tanpa early-stop di semua tier.

In [45]:
# Sel 2: Konfigurasi Dinamis — EDIT DI SINI SAJA, sel lain otomatis mengikuti.
# Alur: CONFIG (bawah) -> override env (ACOS_*) -> validasi -> preset adaptif tier ->
# override eksplisit (STEP1/2_BATCH, FORCE_REBUILD_DATA, RESUME) -> variabel turunan global.
import ast as _ast
CONFIG = {
    # Paksa tier bila deteksi Sel 1 salah / ingin simulasi: AUTO ikuti GPU_TIER Sel 1.
    # SMALL = T4/CPU (minimum + cache); MEDIUM = L4 (menengah + cache); LARGE = MI300X/A100 (besar, from-scratch).
    'GPU_TIER_OVERRIDE': 'AUTO',  # AUTO/SMALL/MEDIUM/LARGE
    # Domain dataset (prefix appsid_quad_*.tsv) & backbone HF (indobert-base-p1).
    'DOMAIN': 'appsid', 'BACKBONE': 'indobert',
    # Grid eksperimen. Total run = len(EPOCHS)*len(RATIOS) + sum(len(EPOCHS)*n_splits).
    # Default [50,75,100] x (3 rasio + 5 + 10 fold) = 54 run.
    'EXPERIMENT_EPOCHS': [50, 75, 100],
    # (train, dev); test = 1-train-dev. Level review_id sehingga klausa 1 ulasan tak bocor.
    'EXPERIMENT_RATIOS': [(0.8, 0.1), (0.7, 0.15), (0.6, 0.2)],  # 80:10:10, 70:15:15, 60:20:20
    # [] untuk menonaktifkan CV; [{'n_splits':5}] untuk smoke-test cepat.
    'EXPERIMENT_CV': [{'n_splits': 5}, {'n_splits': 10}],
    # Eksekusi Sel 9: all = rasio+CV; ratio = hanya split; cv = hanya fold.
    'RUN_MODE': 'all',  # all/ratio/cv
    # True = preview grid tanpa training; False = latih sungguhan. Selalu mulai True.
    'DRY_RUN': True, 'RUN_EPOCHS': None,  # RUN_EPOCHS None = pakai EXPERIMENT_EPOCHS; mis. [50] untuk uji 1 epoch-set
    # Override batch fisik (per-device). None = preset tier; batch efektif = batch x GRAD_ACCUM_STEPS.
    'STEP1_BATCH': None, 'STEP2_BATCH': None,  # mis. 16 / 8 untuk T4
    # LR Step1 (co-extraction) & Step2 (category-sentiment), panjang sekuen BERT, seed, lower-case IndoBERT.
    'STEP1_LR': 2e-5, 'STEP2_LR': 5e-5, 'MAX_SEQ_LENGTH': 128, 'SEED': 42,
    # DO_LOWER_CASE True wajib untuk indobert (vocab lower-case).
    # DRIVE_SYNC True = mirror ke /content/drive/... bila termount; False = lokal saja.
    # FORCE_REBUILD_DATA True = bangun ulang tokenized (lambat); None = ikut tier (LARGE False, SMALL/MEDIUM False + cache).
    # RESUME True = pakai sesi/checkpoint lama; None = ikut tier (LARGE False, SMALL/MEDIUM True).
    'DO_LOWER_CASE': True, 'DRIVE_SYNC': True, 'FORCE_REBUILD_DATA': None, 'RESUME': None,
}
# Helper env: kembalikan default bila variabel tak diset / kosong, agar notebook tetap jalan di semua server.
def _env(n, d=None):
    v = os.environ.get(n)
    return d if v is None or v == '' else v
CONFIG['GPU_TIER_OVERRIDE'] = _env('ACOS_TIER', CONFIG['GPU_TIER_OVERRIDE'])
CONFIG['RUN_MODE'] = _env('ACOS_MODE', CONFIG['RUN_MODE'])
if _env('ACOS_DRY') is not None: CONFIG['DRY_RUN'] = str(_env('ACOS_DRY')).lower() in ('1','true','ya')
if _env('ACOS_EPOCHS') is not None:
    try: CONFIG['RUN_EPOCHS'] = list(_ast.literal_eval(_env('ACOS_EPOCHS')))
    except Exception: pass
if _env('ACOS_BATCH1') is not None:
    try: CONFIG['STEP1_BATCH'] = int(_env('ACOS_BATCH1'))
    except Exception: pass
if _env('ACOS_BATCH2') is not None:
    try: CONFIG['STEP2_BATCH'] = int(_env('ACOS_BATCH2'))
    except Exception: pass
# Turunkan ke variabel global yang dipakai Sel 5/6/7/9 (tokenizer, grid, prepare_all_data, run_all_experiments).
DOMAIN, BACKBONE = CONFIG['DOMAIN'], CONFIG['BACKBONE']
EXPERIMENT_EPOCHS, EXPERIMENT_RATIOS, EXPERIMENT_CV = CONFIG['EXPERIMENT_EPOCHS'], CONFIG['EXPERIMENT_RATIOS'], CONFIG['EXPERIMENT_CV']
# Validasi cepat agar salah ketik ketahuan di awal, bukan setelah berjam-jam training.
assert all(e > 0 for e in EXPERIMENT_EPOCHS), 'EXPERIMENT_EPOCHS harus >0'
assert all(0 < a < 1 and 0 < b < 1 and a + b < 1 for a, b in EXPERIMENT_RATIOS), 'rasio train+dev harus <1'
assert CONFIG['RUN_MODE'] in ('all', 'ratio', 'cv'), 'RUN_MODE all/ratio/cv'

# ======================================================
# PRESET ADAPTIF BERDASARKAN TIER (hasil Sel 1 atau GPU_TIER_OVERRIDE).
# SMALL (T4/CPU, favor hemat VRAM): batch 16/8, accum 4 (efektif 64/32), worker 2, AMP fp16,
#   cache True (pakai ulang tokenized/backbone/checkpoint), resume True.
# MEDIUM (L4): batch 32/24, accum 2 (efektif 64/48), worker 2, AMP fp16, cache True, resume True.
# LARGE (MI300X/A100, kejar throughput): batch 96/64, accum 1, worker 4, AMP bf16 (ROCm) / fp16 (CUDA),
#   from-scratch True, cache False, resume False (folder baru tiap run).
# PATIENCE=0 di semua tier = tanpa early-stop (warisan V5).
# ======================================================
MAX_SEQ_LENGTH, STEP1_LR, STEP2_LR = CONFIG['MAX_SEQ_LENGTH'], CONFIG['STEP1_LR'], CONFIG['STEP2_LR']
SEED, DO_LOWER_CASE = CONFIG['SEED'], CONFIG['DO_LOWER_CASE']
TIER = CONFIG['GPU_TIER_OVERRIDE'] if CONFIG['GPU_TIER_OVERRIDE'] != 'AUTO' else globals().get('GPU_TIER', 'SMALL')
if TIER == 'LARGE':
    STEP1_BATCH_SIZE, STEP2_BATCH_SIZE = 96, 64
    GRAD_ACCUM_STEPS, NUM_WORKERS = 1, 4
    USE_AMP, AMP_DTYPE = True, ('bfloat16' if globals().get('BACKEND') == 'rocm' else 'float16')
    TRAIN_FROM_SCRATCH, USE_MODEL_CACHE = True, False
    FORCE_REBUILD_DATA, RESUME_LAST_SESSION = False, False
    PATIENCE, MIN_EPOCHS_BEFORE_STOP = 0, 5
elif TIER == 'MEDIUM':
    STEP1_BATCH_SIZE, STEP2_BATCH_SIZE = 32, 24
    GRAD_ACCUM_STEPS, NUM_WORKERS = 2, 2
    USE_AMP, AMP_DTYPE = True, 'float16'
    TRAIN_FROM_SCRATCH, USE_MODEL_CACHE = False, True
    FORCE_REBUILD_DATA, RESUME_LAST_SESSION = False, True
    PATIENCE, MIN_EPOCHS_BEFORE_STOP = 0, 5
else:
    STEP1_BATCH_SIZE, STEP2_BATCH_SIZE = 16, 8
    GRAD_ACCUM_STEPS, NUM_WORKERS = 4, 2
    USE_AMP, AMP_DTYPE = True, 'float16'
    TRAIN_FROM_SCRATCH, USE_MODEL_CACHE = False, True
    FORCE_REBUILD_DATA, RESUME_LAST_SESSION = False, True
    PATIENCE, MIN_EPOCHS_BEFORE_STOP = 0, 5
# ======================================================
# Optimasi backend: benchmark cudnn/ROCm bila GPU tersedia (diaktifkan di Sel 3).
# ======================================================
ROCM_BENCHMARK   = True      # torch.backends.cudnn.benchmark = True bila HAS_CUDA

# ======================================================
# MONITORING & LOGGING per-epoch (dibaca HardwareMonitor + ResultSaver Sel 4/8).
# LOG_EVERY_N_STEPS = interval print batch; MONITOR_VRAM = catat VRAM via torch;
# SAVE_HARDWARE_LOG = tulis hardware_log.json tiap run; VRAM_FLUSH_EPOCH = empty_cache() tiap epoch.
# ======================================================
LOG_EVERY_N_STEPS  = 10      # cetak stats setiap N batch
MONITOR_VRAM       = True    # catat VRAM per epoch
SAVE_HARDWARE_LOG  = True    # simpan hardware_log.json
VRAM_FLUSH_EPOCH   = True    # torch.cuda.empty_cache() tiap epoch (penting untuk T4/L4)

# Ringkasan efektif: nilai di bawah inilah yang dipakai sel berikutnya (bukan CONFIG mentah).
# Cek TIER, batch efektif, mode cache/resume, dan RUN_MODE/DRY_RUN sebelum lanjut ke Sel 3.
print('=' * 65)
print(f'KONFIGURASI V5.1 DINAMIS — TIER={TIER} BACKEND={globals().get("BACKEND", "?")}')
print('=' * 65)
print(f'Domain              : {DOMAIN}')
print(f'Backbone            : {BACKBONE}')
print(f'Experiment epochs   : {EXPERIMENT_EPOCHS}')
print(f'Experiment ratios   : {[(int(r[0]*100), int(r[1]*100)) for r in EXPERIMENT_RATIOS]}')
print(f'Experiment CV       : {[c["n_splits"] for c in EXPERIMENT_CV]}-fold')
total_runs = (len(EXPERIMENT_EPOCHS) * len(EXPERIMENT_RATIOS) +
              sum(len(EXPERIMENT_EPOCHS) * c['n_splits'] for c in EXPERIMENT_CV))
print(f'Total runs          : {total_runs}')
print()
if CONFIG['STEP1_BATCH']: STEP1_BATCH_SIZE = int(CONFIG['STEP1_BATCH'])
if CONFIG['STEP2_BATCH']: STEP2_BATCH_SIZE = int(CONFIG['STEP2_BATCH'])
if CONFIG['FORCE_REBUILD_DATA'] is not None: FORCE_REBUILD_DATA = bool(CONFIG['FORCE_REBUILD_DATA'])
if CONFIG['RESUME'] is not None: RESUME_LAST_SESSION = bool(CONFIG['RESUME'])
DRY_RUN, RUN_MODE = bool(CONFIG['DRY_RUN']), CONFIG['RUN_MODE']
RUN_EPOCHS = list(CONFIG['RUN_EPOCHS']) if CONFIG['RUN_EPOCHS'] else list(EXPERIMENT_EPOCHS)
print(f'STEP1_BATCH_SIZE    : {STEP1_BATCH_SIZE}')
print(f'STEP2_BATCH_SIZE    : {STEP2_BATCH_SIZE}')
print(f'GRAD_ACCUM_STEPS    : {GRAD_ACCUM_STEPS}')
print(f'NUM_WORKERS         : {NUM_WORKERS}')
print(f'PATIENCE            : {PATIENCE} (0 = no early stop)')
print(f'TRAIN_FROM_SCRATCH  : {TRAIN_FROM_SCRATCH}')
print(f'USE_MODEL_CACHE     : {USE_MODEL_CACHE}')
print(f'FORCE_REBUILD_DATA  : {FORCE_REBUILD_DATA}')
print(f'RESUME_LAST_SESSION : {RESUME_LAST_SESSION}')
print(f'RUN_MODE/DRY_RUN    : {RUN_MODE}/{DRY_RUN} epochs={RUN_EPOCHS}')
print(f'USE_AMP             : {USE_AMP} ({AMP_DTYPE})')
print(f'ROCM_BENCHMARK      : {ROCM_BENCHMARK}')
print('=' * 65)
# --- V5.1: Drive sync standar ---
DRIVE_SYNC = bool(CONFIG['DRIVE_SYNC'])
GDRIVE_URL = 'https://drive.google.com/drive/folders/1AEzC-dncJAweHPHUPdnFfPHxbsCa83_K'
GDRIVE_ACOS_INDO = '/content/drive/MyDrive/ACOS/ACOS-IndoBERT'
GDRIVE_CANDIDATES = [GDRIVE_ACOS_INDO, '/content/drive/MyDrive/ACOS', '/content/drive/MyDrive/ACOS-ASLI']
GDRIVE_BACKUP_SUBDIR = 'ACOS_V51_BACKUP'
SESSION_REUSE_WINDOW_MIN = 60
FORCE_NEW_SESSION = (TIER == 'LARGE')
if USE_MODEL_CACHE:
    print('Mode cache: tokenized_data/backbone/checkpoint dipakai ulang, tidak rebuild.')
else:
    print('Mode from-scratch: latih dari awal untuk GPU besar.')
print(f'\nDRIVE_SYNC={DRIVE_SYNC} BACKEND={BACKEND} TIER={TIER} AMP_DTYPE={AMP_DTYPE}')
print(f'Drive standar: {GDRIVE_ACOS_INDO} ({GDRIVE_URL})')



KONFIGURASI V5.1 DINAMIS — TIER=SMALL BACKEND=cuda
Domain              : appsid
Backbone            : indobert
Experiment epochs   : [50, 75, 100]
Experiment ratios   : [(80, 10), (70, 15), (60, 20)]
Experiment CV       : [5, 10]-fold
Total runs          : 54

STEP1_BATCH_SIZE    : 16
STEP2_BATCH_SIZE    : 8
GRAD_ACCUM_STEPS    : 4
NUM_WORKERS         : 2
PATIENCE            : 0 (0 = no early stop)
TRAIN_FROM_SCRATCH  : False
USE_MODEL_CACHE     : True
FORCE_REBUILD_DATA  : False
RESUME_LAST_SESSION : True
RUN_MODE/DRY_RUN    : all/True epochs=[50, 75, 100]
USE_AMP             : True (float16)
ROCM_BENCHMARK      : True
Mode cache: tokenized_data/backbone/checkpoint dipakai ulang, tidak rebuild.

DRIVE_SYNC=True BACKEND=cuda TIER=SMALL AMP_DTYPE=float16
Drive standar: /content/drive/MyDrive/ACOS/ACOS-IndoBERT (https://drive.google.com/drive/folders/1AEzC-dncJAweHPHUPdnFfPHxbsCa83_K)


## Sel 3 — Import & Path Setup

In [46]:
# Sel 3: Import semua modul & setup path dua-root (ROBUST V5.1 - Fixed)
import os, sys, time, json, pickle, random, math, importlib, urllib.request
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ============================================================
# 1. DETEKSI ROOT DIREKTORI PROYEK DENGAN VALIDASI LENGKAP
# ============================================================

ACOS_REPO_URL = "https://github.com/haisyamalawwab/ACOS.git"
ACOS_ID_MODULES = [
    "taxonomy", "checkpoint", "cross_val", "experiment_runner",
    "result_saver", "model_wrappers", "build_acos", "tokenize_data", "selftest"
]

# Helper validation functions
def _is_upstream(d):
    """Validasi Extract-Classify-ACOS lengkap (modeling.py + bert_utils/tokenization.py)."""
    return os.path.isfile(os.path.join(d, 'modeling.py')) and os.path.isfile(os.path.join(d, 'bert_utils', 'tokenization.py'))

def _validate_acos_id_complete(indo_path):
    """Validasi semua modul acos_id ada dan tidak kosong."""
    acos_id_dir = os.path.join(indo_path, 'acos_id')
    if not os.path.isdir(acos_id_dir):
        return False, ['acos_id directory missing']
    missing = []
    for m in ACOS_ID_MODULES:
        fpath = os.path.join(acos_id_dir, f"{m}.py")
        if not os.path.isfile(fpath) or os.path.getsize(fpath) == 0:
            missing.append(f"{m}.py")
    return len(missing) == 0, missing

def _cari_base_project():
    """Cari base_project_dir: root yang memuat Extract-Classify-ACOS dan ACOS-IndoBERT LENGKAP."""
    kandidat = [
        "/shared-docker/ACOS",
        "d:/laragon/www/ACOS-ASLI",
        "D:/laragon/www/ACOS-ASLI",
        "/content/drive/MyDrive/ACOS",
        "/content/drive/MyDrive/ACOS-ASLI",
        str(Path.cwd().parent.parent),
        str(Path.cwd().parent),
        str(Path.cwd()),
    ]
    for _c in kandidat:
        p = Path(_c)
        if not p.is_dir():
            continue
        
        extract = p / "Extract-Classify-ACOS"
        indo = p / "ACOS-IndoBERT"
        
        # Validasi struktur dasar
        if not (extract.exists() or indo.exists()):
            continue
        
        # Validasi Extract-Classify-ACOS lengkap (bert_utils harus ada!)
        has_valid_upstream = extract.exists() and _is_upstream(str(extract))
        
        # Validasi acos_id lengkap
        has_valid_indo = False
        if indo.exists():
            is_complete, missing = _validate_acos_id_complete(str(indo))
            has_valid_indo = is_complete
            if not is_complete:
                print(f"⚠️  Skipping {_c}: acos_id incomplete (missing: {missing[:3]}...)")
        
        # Butuh minimal salah satu lengkap
        if has_valid_upstream or has_valid_indo:
            if not has_valid_upstream:
                print(f"⚠️  {_c}: bert_utils missing, will try to clone Extract-Classify-ACOS later")
            return str(p.resolve())
    
    return None

base_project_dir = _cari_base_project()
if base_project_dir is None:
    base_project_dir = os.path.abspath(".")
    print(f"⚠️  No valid project root found, using current directory: {base_project_dir}")

def _cari_indo_root():
    """Folder ACOS-IndoBERT dengan acos_id lengkap."""
    kandidat = [
        os.path.join(base_project_dir, "ACOS-IndoBERT"),
        os.path.abspath("ACOS-IndoBERT"),
        os.path.abspath(os.path.join("..", "ACOS-IndoBERT")),
        os.path.abspath(os.path.join("..", "..", "ACOS-IndoBERT")),
        os.path.abspath("."),
    ]
    for _d in kandidat:
        if os.path.isdir(os.path.join(_d, "acos_id")):
            is_complete, missing = _validate_acos_id_complete(_d)
            if is_complete:
                return str(Path(_d).resolve())
            else:
                print(f"⚠️  Skipping {_d}: incomplete acos_id (missing: {missing[:3]}...)")
    return None

indo_root = _cari_indo_root()
if indo_root is None:
    _target = os.path.join(base_project_dir, "ACOS-IndoBERT")
    print(f"📥 ACOS-IndoBERT belum ada. Menyinkronkan ke {_target} ...")
    _tmp = "/tmp/ACOS_clone_indo"
    os.system(f"rm -rf {_tmp}")
    os.system(f"git clone --depth 1 {ACOS_REPO_URL} {_tmp}")
    _src = os.path.join(_tmp, "ACOS-IndoBERT")
    if os.path.isdir(os.path.join(_src, "acos_id")):
        os.makedirs(_target, exist_ok=True)
        os.system(f'cp -r "{_src}/." "{_target}/"')
        os.system(f"rm -rf {_tmp}")
        indo_root = str(Path(_target).resolve())
        print("✅ ACOS-IndoBERT tersinkron.")
    else:
        indo_root = _target

# ============================================================
# 2. VALIDASI & SYNC MODUL ACOS_ID
# ============================================================
_acos_id_dir = os.path.join(indo_root, "acos_id")
os.makedirs(_acos_id_dir, exist_ok=True)

_missing = [
    f"{_m}.py" for _m in ACOS_ID_MODULES
    if not os.path.isfile(os.path.join(_acos_id_dir, f"{_m}.py"))
    or os.path.getsize(os.path.join(_acos_id_dir, f"{_m}.py")) == 0
]

if _missing:
    print(f"📥 Modul acos_id belum lengkap ({len(_missing)} files). Menyinkronkan...")
    # 1. Coba git pull
    if os.path.isdir(os.path.join(base_project_dir, ".git")):
        os.system(f"git -C '{base_project_dir}' pull origin main")
    
    # 2. Download missing files
    _still_missing = [
        _m for _m in _missing
        if not os.path.isfile(os.path.join(_acos_id_dir, _m))
        or os.path.getsize(os.path.join(_acos_id_dir, _m)) == 0
    ]
    for _fn in _still_missing:
        _url = f"https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/ACOS-IndoBERT/acos_id/{_fn}"
        _target_file = os.path.join(_acos_id_dir, _fn)
        try:
            urllib.request.urlretrieve(_url, _target_file)
            print(f"   ✅ {_fn} berhasil diunduh dari GitHub")
        except Exception as _e:
            print(f"   ✗ Gagal mengunduh {_fn}: {_e}")

# ============================================================
# 3. CARI & VALIDASI EXTRACT-CLASSIFY-ACOS (UPSTREAM ROOT)
# ============================================================
def _prepend_path(p):
    """Paksa p ke posisi terdepan sys.path."""
    try:
        p_str = str(Path(p).resolve())
    except Exception:
        return None
    if not os.path.isdir(p_str):
        return None
    while p_str in sys.path:
        sys.path.remove(p_str)
    sys.path.insert(0, p_str)
    return p_str

# Kandidat upstream dengan prioritas
_up_candidates = [
    os.path.join(base_project_dir, 'Extract-Classify-ACOS'),
    os.path.join(os.path.dirname(indo_root), 'Extract-Classify-ACOS'),
    '/content/Extract-Classify-ACOS',
    '/content/ACOS/Extract-Classify-ACOS',
    '/content/drive/MyDrive/ACOS/Extract-Classify-ACOS',
    '/content/drive/MyDrive/ACOS-ASLI/Extract-Classify-ACOS',
]

upstream_root = None
for _c in _up_candidates:
    if _is_upstream(_c):
        upstream_root = str(Path(_c).resolve())
        print(f"✓ Found valid upstream: {upstream_root}")
        break

# Clone jika belum ada
if upstream_root is None:
    print(f"📥 Extract-Classify-ACOS belum ada. Clone dari {ACOS_REPO_URL} ...")
    _tmpf = '/tmp/ACOS_full'
    os.system(f"rm -rf {_tmpf}")
    os.system(f"git clone --depth 1 {ACOS_REPO_URL} {_tmpf}")
    _srcE = os.path.join(_tmpf, 'Extract-Classify-ACOS')
    _dstE = os.path.join(base_project_dir if os.path.isdir(base_project_dir) else '/content', 'Extract-Classify-ACOS')
    if _is_upstream(_srcE):
        os.makedirs(os.path.dirname(_dstE), exist_ok=True)
        os.system(f'cp -r "{_srcE}" "{_dstE}"')
        print(f'  ✓ Copied: {_srcE} -> {_dstE}')
        if _is_upstream(_dstE):
            upstream_root = _dstE
    os.system(f"rm -rf {_tmpf}")

if upstream_root is None or not _is_upstream(upstream_root):
    raise ModuleNotFoundError(
        f"❌ bert_utils tidak ditemukan!\n"
        f"upstream_root={upstream_root}\n"
        f"Dicoba: {_up_candidates}\n"
        f"Solusi: Upload folder Extract-Classify-ACOS atau git clone {ACOS_REPO_URL}"
    )

# Setup sys.path
_prepend_path(indo_root)
_prepend_path(upstream_root)
_prepend_path(base_project_dir)

# Clear module cache
for _m in list(sys.modules.keys()):
    if _m == "acos_id" or _m.startswith("acos_id."):
        del sys.modules[_m]

print("\n" + "="*65)
print("✅ PATHS CONFIGURED SUCCESSFULLY")
print("="*65)
print(f"  base_project_dir : {base_project_dir}")
print(f"  upstream_root    : {upstream_root}")
print(f"  indo_root        : {indo_root}")
print(f"  modeling.py      : {os.path.isfile(os.path.join(upstream_root, 'modeling.py'))}")
print(f"  bert_utils/      : {os.path.isfile(os.path.join(upstream_root, 'bert_utils', 'tokenization.py'))}")

# ============================================================
# 4. IMPORT MODUL LENGKAP
# ============================================================
print(f"\n{'='*60}")
print("Importing modules...")
print("=" * 60)

# ala V4/V4_1: import upstream dulu, ensure_path validasi 4 berkas kunci, baru bert_utils.
# V4: _prepend indo -> import acos_id.upstream -> ensure_path(acos_root) -> sys.path[0]=Extract-Classify-ACOS.
acos_upstream = importlib.import_module("acos_id.upstream")
try:
    extract_dir = acos_upstream.ensure_path(acos_root=base_project_dir)
except FileNotFoundError:
    print(f"📥 Extract-Classify-ACOS tak valid di {base_project_dir}. Clone dari {ACOS_REPO_URL} ...")
    _tmpE, _okE = '/tmp/ACOS_clone_up', False
    os.system(f"rm -rf {_tmpE}")
    os.system(f"git clone --depth 1 {ACOS_REPO_URL} {_tmpE}")
    for _cand in [os.path.join(base_project_dir, 'Extract-Classify-ACOS'), '/content/Extract-Classify-ACOS', '/content/ACOS/Extract-Classify-ACOS']:
        _srcE = os.path.join(_tmpE, 'Extract-Classify-ACOS')
        if acos_upstream.is_upstream(_srcE):
            os.makedirs(_cand, exist_ok=True)
            os.system(f'cp -r "{_srcE}/." "{_cand}/"')
            _okE = True
            break
    os.system(f"rm -rf {_tmpE}")
    extract_dir = acos_upstream.ensure_path(acos_root=base_project_dir)
upstream_root = extract_dir
_prepend_path(upstream_root)
print(f"  extract_dir valid: {extract_dir}")
from bert_utils.tokenization import BertTokenizer
print("  ✓ BertTokenizer")
from modeling import BertForQuadABSA, CategorySentiClassification
print("  ✓ BertForQuadABSA, CategorySentiClassification")

acos_id = importlib.import_module("acos_id")
print(f"  ✓ acos_id (v{getattr(acos_id, '__version__', '0.2.1')})")

acos_taxonomy = importlib.import_module("acos_id.taxonomy")
print(f"  ✓ taxonomy ({len(getattr(acos_taxonomy, 'CATEGORIES', []))} categories)")

acos_ckpt = importlib.import_module("acos_id.checkpoint")
print("  ✓ checkpoint")

acos_cross_val = importlib.import_module("acos_id.cross_val")
build_with_ratio = acos_cross_val.build_with_ratio
build_kfold_splits = acos_cross_val.build_kfold_splits
ExperimentGrid = acos_cross_val.ExperimentGrid
tokenize_all_splits = acos_cross_val.tokenize_all_splits
print("  ✓ cross_val")

acos_experiment_runner = importlib.import_module("acos_id.experiment_runner")
prepare_all_data = acos_experiment_runner.prepare_all_data
run_all_experiments = acos_experiment_runner.run_all_experiments
aggregate_cv_results = acos_experiment_runner.aggregate_cv_results
print("  ✓ experiment_runner")

acos_result_saver = importlib.import_module("acos_id.result_saver")
ResultSaver = acos_result_saver.ResultSaver
merge_experiment_results = acos_result_saver.merge_experiment_results
print("  ✓ result_saver")

model_wrappers = importlib.import_module("acos_id.model_wrappers")
print("  ✓ model_wrappers")

print(f"\n✅ ALL IMPORTS SUCCESSFUL!")

# ============================================================
# 5. SEED GLOBAL & DIREKTORI OUTPUT
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    if globals().get('ROCM_BENCHMARK', True):
        torch.backends.cudnn.benchmark = True

data_dir        = os.path.join(indo_root, "data", "Apps-ACOS")
processed_dir   = os.path.join(data_dir, "processed")
tokenized_base  = os.path.join(indo_root, "tokenized_data")
results_base    = os.path.join(indo_root, "results")
experiments_dir = os.path.join(results_base, "experiments")
bert_cache_dir  = os.path.join(indo_root, "backbones", "indobert_base_p1")

os.makedirs(experiments_dir, exist_ok=True)

print(f"\nDirectory paths configured:")
print(f"  data_dir        : {data_dir}")
print(f"  tokenized_base  : {tokenized_base}")
print(f"  experiments_dir : {experiments_dir}")

# --- V5.1: Drive sync helper ---
import shutil
GDRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')
GDRIVE_BACKUP_DIR = None
if GDRIVE_MOUNTED:
    GDRIVE_BACKUP_DIR = os.path.join('/content/drive/MyDrive', globals().get('GDRIVE_BACKUP_SUBDIR', 'ACOS_V51_BACKUP'))
    os.makedirs(GDRIVE_BACKUP_DIR, exist_ok=True)
    print(f'\n✓ Drive mounted: {GDRIVE_BACKUP_DIR}')
else:
    print('\n✓ Running locally (no Drive sync)')

def sync_to_gdrive(source_path, target_subpath=''):
    """Mirror file/folder ke Google Drive bila mounted."""
    if not globals().get('DRIVE_SYNC', True) or not GDRIVE_MOUNTED or not GDRIVE_BACKUP_DIR:
        return None
    try:
        dest_dir = os.path.join(GDRIVE_BACKUP_DIR, target_subpath) if target_subpath else GDRIVE_BACKUP_DIR
        os.makedirs(dest_dir, exist_ok=True)
        if os.path.isdir(source_path):
            dest = os.path.join(dest_dir, os.path.basename(source_path.rstrip('/')))
            if os.path.exists(dest):
                shutil.rmtree(dest)
            shutil.copytree(source_path, dest)
            return dest
        else:
            shutil.copy2(source_path, dest_dir)
            return os.path.join(dest_dir, os.path.basename(source_path))
    except Exception as e:
        print(f'⚠️  sync_to_gdrive gagal: {e}')
        return None

print(f"\n✅ Setup complete! Ready for training.")


⚠️  /content: bert_utils missing, will try to clone Extract-Classify-ACOS later
📥 Extract-Classify-ACOS belum ada. Clone dari https://github.com/haisyamalawwab/ACOS.git ...
  ✓ Copied: /tmp/ACOS_full/Extract-Classify-ACOS -> /content/Extract-Classify-ACOS

✅ PATHS CONFIGURED SUCCESSFULLY
  base_project_dir : /content
  upstream_root    : /content/Extract-Classify-ACOS
  indo_root        : /content/ACOS-IndoBERT
  modeling.py      : True
  bert_utils/      : True

Importing modules...


ModuleNotFoundError: No module named 'bert_utils'

## Sel 4 — HardwareMonitor

Kelas untuk monitoring real-time:
- VRAM (used/total/%) per epoch
- Waktu per epoch dan per batch
- GPU utilization via `rocm-smi` atau `nvidia-smi`
- Menyimpan `hardware_log.json` di setiap folder run

In [ ]:
# Sel 4: HardwareMonitor — monitoring real-time GPU AMD MI300X
import threading, time, subprocess, json
from dataclasses import dataclass, field, asdict
from typing import List, Optional


def _fmt_dur(sec: float) -> str:
    if sec < 60: return f'{sec:.1f}s'
    m, s = divmod(int(sec), 60)
    if m < 60: return f'{m}m{s:02d}s'
    h, m = divmod(m, 60)
    return f'{h}h{m:02d}m'


@dataclass
class EpochStats:
    epoch: int
    phase: str = ''
    run_id: str = ''
    epoch_start: float = 0.0
    epoch_end: float = 0.0
    duration_sec: float = 0.0
    loss: float = 0.0
    f1: float = 0.0
    vram_used_gb: float = 0.0
    vram_total_gb: float = 0.0
    vram_pct: float = 0.0
    gpu_util_pct: Optional[float] = None
    batch_per_sec: float = 0.0
    samples_per_sec: float = 0.0


@dataclass
class RunStats:
    run_id: str
    started_at: str = ''
    finished_at: str = ''
    total_duration_sec: float = 0.0
    config: dict = field(default_factory=dict)
    epochs: List[EpochStats] = field(default_factory=list)
    hardware: dict = field(default_factory=dict)
    metrics: dict = field(default_factory=dict)


class HardwareMonitor:
    """Monitor VRAM, waktu, dan GPU utilization per epoch.

    Usage::
        monitor = HardwareMonitor(log_dir=result_dir)
        monitor.start_run('split_801010_ep50', config=cfg)
        for epoch in range(1, NUM_EPOCHS + 1):
            monitor.start_epoch(epoch, phase='step1')
            # ... training ...
            stats = monitor.end_epoch(loss=0.42, f1=96.5,
                                      n_batches=627, batch_size=96)
        monitor.end_run(metrics={'step1_f1': 97.1})
    """

    def __init__(self, log_dir: str, device: int = 0):
        self.log_dir = log_dir
        self.device  = device
        self._run: Optional[RunStats] = None
        self._epoch_start = 0.0
        self._current_epoch = 0
        self._current_phase = ''
        os.makedirs(log_dir, exist_ok=True)

    def start_run(self, run_id: str, config: dict = None):
        hw = self._collect_hardware()
        self._run = RunStats(
            run_id=run_id,
            started_at=datetime.now().isoformat(),
            config=config or {},
            hardware=hw,
        )
        self._print_header(run_id, hw)

    def start_epoch(self, epoch: int, phase: str = 'step1'):
        self._epoch_start = time.time()
        self._current_epoch = epoch
        self._current_phase = phase

    def end_epoch(self, loss: float, f1: float,
                  n_batches: int = 0, batch_size: int = 96) -> EpochStats:
        t_end = time.time()
        dur   = t_end - self._epoch_start
        vram  = self._get_vram()
        gpu_u = self._get_gpu_util()
        bps   = n_batches / dur if dur > 0 else 0
        sps   = (n_batches * batch_size) / dur if dur > 0 else 0

        stats = EpochStats(
            epoch=self._current_epoch,
            phase=self._current_phase,
            run_id=self._run.run_id if self._run else '',
            epoch_start=self._epoch_start,
            epoch_end=t_end,
            duration_sec=round(dur, 2),
            loss=round(loss, 6),
            f1=round(f1, 4),
            vram_used_gb=vram['used_gb'],
            vram_total_gb=vram['total_gb'],
            vram_pct=vram['pct'],
            gpu_util_pct=gpu_u,
            batch_per_sec=round(bps, 2),
            samples_per_sec=round(sps, 1),
        )
        if self._run:
            self._run.epochs.append(stats)
        self._print_epoch(stats)
        return stats

    def end_run(self, metrics: dict = None):
        if not self._run: return
        self._run.finished_at = datetime.now().isoformat()
        self._run.total_duration_sec = sum(e.duration_sec for e in self._run.epochs)
        self._run.metrics = metrics or {}
        self._save_log()
        self._print_footer()

    def vram_str(self) -> str:
        v = self._get_vram()
        return f"{v['used_gb']:.2f}/{v['total_gb']:.2f} GB ({v['pct']:.1f}%)"

    def _get_vram(self) -> dict:
        if not torch.cuda.is_available():
            return {'used_gb': 0.0, 'total_gb': 0.0, 'pct': 0.0}
        used  = torch.cuda.memory_allocated(self.device)
        total = torch.cuda.get_device_properties(self.device).total_memory
        ug = used / 1024**3
        tg = total / 1024**3
        return {'used_gb': round(ug, 3), 'total_gb': round(tg, 3),
                'pct': round(ug / tg * 100, 2) if tg > 0 else 0.0}

    def _get_gpu_util(self) -> Optional[float]:
        backend = globals().get('BACKEND', 'cuda')
        try:
            if backend == 'rocm' or HAS_ROCM:
                out = subprocess.check_output(
                    ['rocm-smi', '--showuse', '--json'],
                    stderr=subprocess.DEVNULL, timeout=5, text=True)
                for card, info in json.loads(out).items():
                    u = info.get('GPU use (%)', None)
                    if u is not None: return float(u)
            else:
                out = subprocess.check_output(
                    ['nvidia-smi', '--query-gpu=utilization.gpu',
                     '--format=csv,noheader,nounits'],
                    stderr=subprocess.DEVNULL, timeout=5, text=True)
                return float(out.strip().split('\n')[0])
        except Exception:
            pass
        return None

    def _collect_hardware(self) -> dict:
        hw = {'timestamp': datetime.now().isoformat(),
              'torch': torch.__version__, 'cuda': HAS_CUDA, 'rocm': HAS_ROCM}
        if HAS_CUDA:
            p = torch.cuda.get_device_properties(0)
            hw['gpu_name'] = p.name
            hw['vram_total_gb'] = round(p.total_memory / 1024**3, 3)
            hw['sm_count'] = p.multi_processor_count
        return hw

    def _save_log(self):
        if not self._run: return
        path = os.path.join(self.log_dir, 'hardware_log.json')
        with open(path, 'w', encoding='utf-8') as fh:
            json.dump(asdict(self._run), fh, indent=2, ensure_ascii=False, default=str)

    def _print_header(self, run_id, hw):
        print(f"\n{'='*72}")
        print(f"  RUN    : {run_id}")
        print(f"  GPU    : {hw.get('gpu_name','CPU')} | "
              f"VRAM: {hw.get('vram_total_gb',0):.2f} GB | ROCm: {hw.get('rocm',False)}")
        print(f"  Start  : {self._run.started_at}")
        print(f"{'='*72}")
        print(f"  {'Ep':>4} {'Phase':>6} {'Loss':>9} {'F1%':>7} "
              f"{'VRAM':>14} {'Util%':>6} {'Samp/s':>8} {'Dur':>8}")
        print(f"  {'-'*4} {'-'*6} {'-'*9} {'-'*7} "
              f"{'-'*14} {'-'*6} {'-'*8} {'-'*8}")

    def _print_epoch(self, s: EpochStats):
        vram_str = f'{s.vram_used_gb:.2f}/{s.vram_total_gb:.2f}G'
        util_str = f'{s.gpu_util_pct:.0f}%' if s.gpu_util_pct is not None else 'N/A'
        print(f"  {s.epoch:>4} {s.phase:>6} {s.loss:>9.4f} {s.f1:>7.2f} "
              f"{vram_str:>14} {util_str:>6} {s.samples_per_sec:>8.0f} {_fmt_dur(s.duration_sec):>8}")

    def _print_footer(self):
        total = self._run.total_duration_sec if self._run else 0
        m = self._run.metrics if self._run else {}
        print(f"  {'='*72}")
        print(f"  Selesai : {self._run.finished_at}")
        print(f"  Durasi  : {_fmt_dur(total)}")
        print(f"  Metrics : Step1 F1={m.get('step1_f1','?')} | Step2 F1={m.get('step2_f1','?')}")
        print(f"{'='*72}\n")


print('HardwareMonitor: OK')
print(f'Contoh: monitor = HardwareMonitor(log_dir=result_dir)')


## Sel 5 — Backbone IndoBERT & Tokenizer

In [ ]:
# Sel 5: Siapkan backbone IndoBERT & tokenizer
from acos_id.checkpoint import prepare_backbone

print('Mempersiapkan backbone IndoBERT...')
backbone_report = prepare_backbone(BACKBONE, bert_cache_dir)
rk = backbone_report.get('rekey', {})
print(f"  Rekey: {rk.get('dilewati', False) and 'dilewati' or 'selesai'}")

tokenizer = BertTokenizer.from_pretrained(bert_cache_dir, do_lower_case=DO_LOWER_CASE)
print(f'  Tokenizer vocab: {len(tokenizer.vocab):,} token')
print()

# Validasi dataset
print('Validasi dataset:')
for split in ('train', 'dev', 'test'):
    fpath = os.path.join(data_dir, f'appsid_quad_{split}.tsv')
    if os.path.exists(fpath):
        n = sum(1 for _ in open(fpath, encoding='utf-8'))
        print(f'  {split:5s}: {n:7,} baris  [OK]')
    else:
        print(f'  {split:5s}: MISSING! {fpath}')

# Validasi backbone files
print()
print('Validasi backbone files:')
for f in ('config.json', 'pytorch_model.bin', 'vocab.txt'):
    fpath = os.path.join(bert_cache_dir, f)
    exists = os.path.exists(fpath)
    sz = os.path.getsize(fpath) / 1024**2 if exists else 0
    print(f'  {f:25s}: {"OK" if exists else "MISSING"} ({sz:.1f} MB)')

print()
print('Backbone & tokenizer: OK')

## Sel 5b — EDA + Visualisasi (ala V4, via `acos_id.eda`)

Tabel statistik + 4 PNG: distribusi split, kategori & sentimen, panjang teks & implisit, heatmap kategori x sentimen. Hasil disimpan di `csv/` dan `plots/` sesi serta dimirror ke Drive.


In [ ]:
# Sel 5b: EDA komprehensif
from acos_id.eda import analyze_and_plot_eda_id
from acos_id.session import session_dirs_from_root as _sdir
from IPython.display import display, Image
import pandas as pd

eda_session = _sdir(os.path.join(experiments_dir, '_eda'))
df_stats, df_records = analyze_and_plot_eda_id(data_dir, domain=DOMAIN, output_plots_dir=eda_session['plots'], output_csv_dir=eda_session['csv'])
if df_stats is not None and not df_stats.empty:
    print(df_stats.to_string(index=False))
    display(df_stats)
    print(f'\nSampel quad: {len(df_records):,} baris')
    display(df_records.head(10))
    for png in sorted(__import__('glob').glob(os.path.join(eda_session['plots'], '*.png'))):
        print(png)
        try: display(Image(png, width=800))
        except Exception: pass
    sync_to_gdrive(eda_session['root'], 'eda')
else:
    print('EDA dilewati: dataset belum tersedia. Jalankan persiapan data dulu atau cek data_dir.')


## Sel 6 — ExperimentGrid (Preview 54 Run)

Jalankan sel ini untuk **melihat semua rencana eksperimen tanpa training**.
Hanya tampilan — tidak ada data yang diproses.

In [ ]:
# Sel 6: Preview semua kombinasi eksperimen
grid = ExperimentGrid()
grid.add_epochs(EXPERIMENT_EPOCHS)
grid.add_ratios(EXPERIMENT_RATIOS)
for cv_cfg in EXPERIMENT_CV:
    grid.add_cv(n_splits=cv_cfg['n_splits'])

grid.print_summary()
print(f'\nTotal: {len(grid)} run')
n_ratio = len(EXPERIMENT_EPOCHS) * len(EXPERIMENT_RATIOS)
n_cv    = sum(len(EXPERIMENT_EPOCHS) * c['n_splits'] for c in EXPERIMENT_CV)
print(f'  Split-ratio: {n_ratio} run')
print(f'  CV total   : {n_cv} run')
print()
# Estimasi kasar: 50 epoch sekitar 4 jam per run di MI300X
est_hr_50  = len(grid) * 4
print(f'Estimasi durasi (semua, 50 ep per run ~4 jam): {est_hr_50} jam = {est_hr_50/24:.1f} hari')
print()
print('Tips: Gunakan mode="ratio" atau mode="cv" di Sel 9 untuk subset.')
# --- V5.1: grafik komposisi grid ---
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    labels = ['ratio', 'cv']; vals = [n_ratio, n_cv]
    fig, ax = plt.subplots(figsize=(6,3.5))
    ax.bar(labels, vals, color=['#2b5c8f','#d95f02'])
    ax.set_title('Komposisi run eksperimen'); ax.set_ylabel('jumlah run')
    for x, v in zip(labels, vals): ax.text(x, v, str(v), ha='center', va='bottom', fontweight='bold')
    plt.tight_layout(); p = os.path.join(experiments_dir, '_grid_composition.png'); plt.savefig(p, dpi=150); plt.close(); print(p)
    sync_to_gdrive(p, '')
except Exception as e: print(f'grafik grid dilewati: {e}')



## Sel 7 — Persiapan Data (Semua Split + Fold, 1x)

> **Jalankan 1x saja.** Hasilnya di-cache.
> Pemanggilan berikutnya skip file yang sudah ada (`force_rebuild=False`).

**Waktu estimasi: 30–60 menit (1x)**

In [ ]:
# Sel 7: Build semua TSV mentah + tokenisasi (1x, cached)
t0 = time.time()

prepared = prepare_all_data(
    indo_root=indo_root,
    tokenizer=tokenizer,
    ratio_list=EXPERIMENT_RATIOS,
    cv_configs=EXPERIMENT_CV,
    seed=SEED,
    force_rebuild=globals().get('FORCE_REBUILD_DATA', False),
)

dur = time.time() - t0
print(f'\nTotal waktu persiapan data: {_fmt_dur(dur)}')
print('Semua data siap untuk training.')

## Sel 8 — Training Adapter Function

Fungsi `train_one_run(cfg)` adalah adapter antara `ExperimentGrid`
dan training loop Step 1 + Step 2.
Setiap panggilan menjalankan 1 kombinasi eksperimen dari awal.

> **Catatan:** Sel ini mendefinisikan fungsi — belum menjalankan training.
> Training dimulai di Sel 9.

In [ ]:
# Sel 8: Adapter — 1 run eksperimen = Step 1 + Step 2
from acos_id.session import session_dirs_from_root
from torch.utils.data import DataLoader


def train_one_run(cfg: dict) -> dict:
    """Jalankan 1 run lengkap (Step 1 + Step 2) dari ExperimentGrid.

    Parameter cfg (dari ExperimentGrid):
      cfg['epochs']        = NUM_EPOCHS untuk run ini
      cfg['tokenized_dir'] = folder tokenized yang dipakai
      cfg['result_dir']    = folder output run ini
      cfg['run_id']        = nama unik run ini
      cfg['seed']          = random seed
    """
    run_id  = cfg['run_id']
    num_ep  = cfg['epochs']
    tok_dir = cfg['tokenized_dir']
    res_dir = cfg['result_dir']
    seed    = cfg.get('seed', SEED)

    if not os.path.isdir(tok_dir):
        raise FileNotFoundError(f'tokenized_dir tidak ada: {tok_dir}')

    # Setup session dirs + ResultSaver V5.1 (komprehensif ala V4)
    from acos_id.result_saver import ResultSaver
    sess = session_dirs_from_root(res_dir)
    saver = ResultSaver(run_dir=res_dir)
    saver.init(config={'run_id': run_id, 'epochs': num_ep, 'seed': seed, 'backend': globals().get('BACKEND','cuda'), 'amp': USE_AMP, 'amp_dtype': AMP_DTYPE})

    # Hardware monitor
    monitor = HardwareMonitor(log_dir=res_dir, device=0)
    monitor.start_run(run_id, config=cfg)

    # AMP scaler
    scaler = None
    if USE_AMP and HAS_CUDA:
        scaler = torch.cuda.amp.GradScaler()

    # ================================================================
    # STEP 1: BertForQuadABSA (co-extraction aspek & opini)
    # ================================================================
    def load_loader(split, batch_size, shuffle):
        path = os.path.join(tok_dir, f'appsid_{split}_quad_bert.tsv')
        dataset = model_wrappers.load_quad_tsv_dataset(
            path, tokenizer, max_seq_length=MAX_SEQ_LENGTH)
        return DataLoader(dataset, batch_size=batch_size,
                          shuffle=shuffle, num_workers=NUM_WORKERS)

    train_loader = load_loader('train', STEP1_BATCH_SIZE, True)
    dev_loader   = load_loader('dev',   STEP1_BATCH_SIZE, False)

    model = BertForQuadABSA.from_pretrained(
        bert_cache_dir,
        num_labels=acos_taxonomy.num_labels_step1(),
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=STEP1_LR)

    best_f1_s1 = 0.0
    best_ep_s1 = 0

    print(f'\n  [Step 1] {num_ep} epoch | batch={STEP1_BATCH_SIZE} | '
          f'AMP={USE_AMP}({AMP_DTYPE}) | PATIENCE={PATIENCE}')

    for epoch in range(1, num_ep + 1):
        monitor.start_epoch(epoch, phase='step1')
        model.train()
        total_loss = 0.0
        n_batches  = 0
        t_ep = time.time()

        for step, batch in enumerate(train_loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()

            if USE_AMP and scaler:
                dtype = torch.bfloat16 if AMP_DTYPE == 'bfloat16' else torch.float16
                with torch.cuda.amp.autocast(dtype=dtype):
                    outputs = model(**batch)
                    loss = outputs['loss'] / GRAD_ACCUM_STEPS
                scaler.scale(loss).backward()
                if (step + 1) % GRAD_ACCUM_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
            else:
                outputs = model(**batch)
                loss = outputs['loss'] / GRAD_ACCUM_STEPS
                loss.backward()
                if (step + 1) % GRAD_ACCUM_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            total_loss += loss.item() * GRAD_ACCUM_STEPS
            n_batches  += 1

            if LOG_EVERY_N_STEPS > 0 and (step + 1) % LOG_EVERY_N_STEPS == 0:
                elapsed = time.time() - t_ep
                sps = (n_batches * STEP1_BATCH_SIZE) / elapsed
                print(f'    ep{epoch:3d} step{step+1:4d}/{len(train_loader)} '
                      f'loss={loss.item():.4f} VRAM={monitor.vram_str()} '
                      f'samp/s={sps:.0f}', end='\r')

        # Evaluasi dev
        eval_results = model_wrappers.compute_extraction_metrics(
            model, dev_loader, DEVICE)
        f1 = eval_results['f1']
        prec = eval_results['precision']
        rec = eval_results['recall']

        avg_loss = total_loss / max(n_batches, 1)
        print()  # newline setelah progress bar
        monitor.end_epoch(avg_loss, f1, n_batches, STEP1_BATCH_SIZE)
        try:
            ev = model_wrappers.compute_extraction_metrics(model, dev_loader, DEVICE)
            saver.log_epoch_step1(epoch=epoch, loss=avg_loss, f1=f1, tp=0, fp=0, fn=0, vram_gb=monitor._get_vram()['used_gb'], duration_sec=0.0, samples_per_sec=0.0)
        except Exception: pass

        if f1 > best_f1_s1:
            best_f1_s1 = f1
            best_ep_s1 = epoch
            ckpt_path = os.path.join(sess['checkpoints'], 'step1_best')
            model.save_pretrained(ckpt_path)
            print(f'    >> Best checkpoint: ep{epoch} F1={f1:.2f}%')

        # PATIENCE = 0 di notebook ini — early stopping TIDAK AKTIF
        # Blok ini tetap ada untuk kompatibilitas tapi tidak pernah terpicu
        if PATIENCE > 0:
            epochs_since_best = epoch - best_ep_s1
            if epoch >= MIN_EPOCHS_BEFORE_STOP and epochs_since_best >= PATIENCE:
                print(f'    >> Early stop ep{epoch} (tidak terpicu karena PATIENCE=0)')
                break

        if VRAM_FLUSH_EPOCH:
            torch.cuda.empty_cache()

    print(f'\n  [Step 1] SELESAI | Best F1={best_f1_s1:.2f}% di ep{best_ep_s1}')

    # ================================================================
    # STEP 2: Klasifikasi Category + Sentiment
    # (Implementasi penuh di sini — serupa Step 1 tapi pakai pair.tsv)
    # ================================================================
    best_f1_s2 = 0.0
    # TODO: Implementasi penuh Step 2
    # Prinsip: ganti load_loader ke pair.tsv, model ke ClassificationModel
    print(f'  [Step 2] TODO — best_f1_s2={best_f1_s2:.2f}%')

    try:
        saver.finalize_step1(best_epoch=best_ep_s1, best_f1=best_f1_s1)
        saver.save_step1_plots()
        saver.write_final_report(extra_info={'run_id': run_id})
    except Exception as e: print(f'ResultSaver dilewati: {e}')
    monitor.end_run(metrics={'step1_f1': round(best_f1_s1, 4), 'step1_best_epoch': best_ep_s1, 'step2_f1': round(best_f1_s2, 4)})
    try:
        from acos_id.session import write_session_manifest
        write_session_manifest(sess, {'run_id': run_id, 'backend': globals().get('BACKEND','cuda')})
    except Exception: pass
    try:
        if 'sync_to_gdrive' in globals(): sync_to_gdrive(res_dir, 'runs')
    except Exception: pass

    torch.cuda.empty_cache()
    return {'step1_f1': best_f1_s1, 'step2_f1': best_f1_s2}


print('train_one_run: terdefinisi.')
print('Fungsi siap. Training dimulai di Sel 9.')


## Sel 9 — Run Semua Eksperimen (Otomatis, 1-per-1)

> **Estimasi total: 5–8 hari untuk 54 run (50–100 epoch per run)**

Parameter `mode`:
- `'all'` — semua 54 run
- `'ratio'` — hanya 9 run split-ratio
- `'cv'` — hanya 45 run cross-validation

Set `DRY_RUN = True` untuk preview urutan tanpa training.

In [ ]:
# Sel 9: Jalankan semua eksperimen — baca CONFIG Sel 2 (tetap bisa override di sini)
DRY_RUN = globals().get('DRY_RUN', True)
RUN_MODE = globals().get('RUN_MODE', 'all')
RUN_EPOCHS = globals().get('RUN_EPOCHS', EXPERIMENT_EPOCHS)
print(f'KONTROL: mode={RUN_MODE} dry={DRY_RUN} epochs={RUN_EPOCHS} (ubah di Sel 2 CONFIG)')

t_start = time.time()

all_results = run_all_experiments(
    train_fn=train_one_run,
    indo_root=indo_root,
    epochs_list=RUN_EPOCHS,
    ratio_list=EXPERIMENT_RATIOS,
    cv_configs=EXPERIMENT_CV,
    seed=SEED,
    mode=RUN_MODE,
    dry_run=DRY_RUN,
)

total_dur = time.time() - t_start
print(f'\nWaktu total: {_fmt_dur(total_dur)}')
print(f'Status: {"DRY RUN" if DRY_RUN else f"{len(all_results)} run selesai"}')

## Sel 10 — Agregasi & Perbandingan Hasil

In [ ]:
# Sel 10: Ringkasan & perbandingan semua hasil
if not all_results:
    print('Belum ada hasil. Jalankan Sel 9 dengan DRY_RUN=False.')
else:
    ok = [r for r in all_results if r.get('status') == 'OK']
    err  = [r for r in all_results if r.get('status') == 'ERROR']
    skip = [r for r in all_results if r.get('status') == 'SKIP_NO_DATA']
    print(f'Total run: {len(all_results)} | OK: {len(ok)} | Error: {len(err)} | Skip: {len(skip)}')

    if ok:
        rows = []
        for r in ok:
            m = r.get('metrics', {})
            split_str = (
                f"{int(r.get('train_ratio',0)*100)}:{int(r.get('dev_ratio',0)*100)}"
                if r['type'] == 'ratio'
                else f"cv{r.get('n_splits','?')}-f{r.get('fold_idx','?')}"
            )
            rows.append({
                'run_id'    : r['run_id'],
                'type'      : r['type'],
                'epochs'    : r['epochs'],
                'split'     : split_str,
                'step1_f1'  : m.get('step1_f1', 0),
                'step2_f1'  : m.get('step2_f1', 0),
                'dur_min'   : round(r.get('duration_sec', 0) / 60, 1),
            })

        df = pd.DataFrame(rows).sort_values('step1_f1', ascending=False)
        print('\nTop 10 berdasarkan Step1 F1:')
        print(df.head(10).to_string(index=False))

        # Agregasi CV
        for n_splits in [5, 10]:
            agg = aggregate_cv_results(all_results, n_splits=n_splits)
            cv_key = f'cv_{n_splits}fold'
            if agg.get(cv_key):
                print(f'\nCV {n_splits}-Fold — mean +/- std:')
                df_cv = pd.DataFrame(agg[cv_key].values())
                cols = ['epoch', 'step1_f1_mean', 'step1_f1_std',
                        'step2_f1_mean', 'step2_f1_std', 'n_folds_completed']
                print(df_cv[[c for c in cols if c in df_cv.columns]].to_string(index=False))

        # Simpan tabel
        summary_path = os.path.join(experiments_dir, 'final_results_summary.csv')
        df.to_csv(summary_path, index=False)
        print(f'\nRingkasan disimpan: {summary_path}')
# --- V5.1: grafik perbandingan + mirror Drive ---
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    top = df.head(10).sort_values('step1_f1')
    fig, ax = plt.subplots(figsize=(8,4.5))
    ax.barh(top['run_id'].astype(str), top['step1_f1'], color='#2b5c8f')
    ax.set_title('Top-10 run Step1 F1'); ax.set_xlabel('F1')
    plt.tight_layout(); pp = os.path.join(experiments_dir, 'top10_step1_f1.png'); plt.savefig(pp, dpi=150); plt.close(); print(pp)
    sync_to_gdrive(summary_path, ''); sync_to_gdrive(pp, '')
except Exception as e: print(f'grafik agregasi dilewati: {e}')



## Sel 11 — Hardware Summary per Run

Baca semua `hardware_log.json` dan tampilkan tabel penggunaan resource:
VRAM max, VRAM rata-rata, GPU utilization, samples/sec, durasi.

In [ ]:
# Sel 11: Ringkasan hardware semua run
import glob

hw_logs = sorted(glob.glob(os.path.join(experiments_dir, '*', 'hardware_log.json')))
print(f'Hardware logs ditemukan: {len(hw_logs)}')

hw_rows = []
for log_path in hw_logs:
    try:
        with open(log_path, encoding='utf-8') as fh:
            data = json.load(fh)
        eps = data.get('epochs', [])
        if eps:
            vram_vals = [e['vram_used_gb'] for e in eps if e.get('vram_used_gb', 0) > 0]
            util_vals = [e['gpu_util_pct'] for e in eps if e.get('gpu_util_pct') is not None]
            sps_vals  = [e['samples_per_sec'] for e in eps if e.get('samples_per_sec', 0) > 0]
            hw_rows.append({
                'run_id'       : data.get('run_id', '?'),
                'dur_min'      : round(data.get('total_duration_sec', 0) / 60, 1),
                'epochs_done'  : len(eps),
                'vram_max_gb'  : round(max(vram_vals), 3) if vram_vals else 0,
                'vram_avg_gb'  : round(sum(vram_vals) / len(vram_vals), 3) if vram_vals else 0,
                'vram_pct_avg' : round(sum(e['vram_pct'] for e in eps) / len(eps), 2) if eps else 0,
                'gpu_util_avg' : round(sum(util_vals) / len(util_vals), 1) if util_vals else None,
                'samp_s_avg'   : round(sum(sps_vals) / len(sps_vals), 0) if sps_vals else 0,
                'step1_f1'     : data.get('metrics', {}).get('step1_f1', None),
            })
    except Exception as e:
        print(f'  Gagal baca {log_path}: {e}')

if hw_rows:
    df_hw = pd.DataFrame(hw_rows).sort_values('step1_f1', ascending=False)
    print('\nHardware Usage per Run:')
    print(df_hw.to_string(index=False))

    hw_summary_path = os.path.join(experiments_dir, 'hardware_summary.csv')
    df_hw.to_csv(hw_summary_path, index=False)
    print(f'\nHardware summary disimpan: {hw_summary_path}')
else:
    print('Belum ada hardware log (jalankan training terlebih dahulu).')

# Ringkasan sesi total
SESSION_END = datetime.now()
print(f'\nSesi: {SESSION_START.strftime("%Y-%m-%d %H:%M:%S")} s/d {SESSION_END.strftime("%H:%M:%S")}')
print(f'Durasi sesi: {_fmt_dur((SESSION_END - SESSION_START).total_seconds())}')
# --- V5.1: plot hardware + REPORT_INDEX + final Drive sync (ala V4) ---
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    if 'df_hw' in globals() and not df_hw.empty:
        fig, ax = plt.subplots(figsize=(8,4))
        ax.scatter(df_hw['vram_max_gb'], df_hw['step1_f1'], c='#d95f02')
        ax.set_xlabel('VRAM max (GB)'); ax.set_ylabel('Step1 F1'); ax.set_title('VRAM vs F1 per run'); ax.grid(alpha=0.3)
        plt.tight_layout(); hp = os.path.join(experiments_dir, 'vram_vs_f1.png'); plt.savefig(hp, dpi=150); plt.close(); print(hp)
except Exception as e: print(f'grafik hardware dilewati: {e}')
try:
    from acos_id.session import write_report_index, write_session_manifest, session_dirs_from_root as _sr
    _sess = _sr(experiments_dir)
    print(write_report_index(_sess))
    sync_to_gdrive(experiments_dir, '')
    print(f"Drive mounted: {GDRIVE_MOUNTED} -> {GDRIVE_BACKUP_DIR}")
    print('Lanjut di server lain: pull repo + download folder backup Drive ini, atau copy experiments/.')
except Exception as e: print(f'final sync dilewati: {e}')

